<a href="https://colab.research.google.com/github/Amuw9/cgitraining/blob/main/DAY06_Assign_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install -U cohere


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 370.5/370.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 20.0 MB/s eta 0:00:00


In [4]:
import os
os.environ['COHERE_API_KEY'] = "swQ13uyNUDG96tKHDkxVn03VUcChRYweU2TWBqMf"

In [5]:
import os
import cohere

# Create Cohere client
co = cohere.ClientV2(
    api_key=os.environ["COHERE_API_KEY"]
)

# User's search query
query = "What are the health and fitness benefits for employees?"

# Imagine these came from Elasticsearch, vector DB, or another
# initial retrieval system.
documents = [
    "Employees can claim reimbursement for approved business travel expenses.",

    "Employees working remotely from another country must coordinate "
    "with their manager and maintain core working hours.",

    "Our health and wellness program includes gym memberships, "
    "on-site yoga classes, mental wellness programs, and health insurance.",

    "Employees receive formal performance reviews twice a year "
    "and informal quarterly check-ins.",

    "The company provides flexible working hours and remote-work options "
    "for eligible employees."
]

# Send retrieved documents to Cohere Rerank
response = co.rerank(
    model="rerank-v4.0-pro",
    query=query,
    documents=documents,
    top_n=3
)

# Display results
print("\n=== RERANKED RESULTS ===\n")

for rank, result in enumerate(response.results, start=1):
    print(f"Rank       : {rank}")
    print(f"Score      : {result.relevance_score:.6f}")
    print(f"Document # : {result.index}")
    print(f"Document   : {documents[result.index]}")
    print("-" * 80)



=== RERANKED RESULTS ===

Rank       : 1
Score      : 0.733814
Document # : 2
Document   : Our health and wellness program includes gym memberships, on-site yoga classes, mental wellness programs, and health insurance.
--------------------------------------------------------------------------------
Rank       : 2
Score      : 0.386393
Document # : 4
Document   : The company provides flexible working hours and remote-work options for eligible employees.
--------------------------------------------------------------------------------
Rank       : 3
Score      : 0.302062
Document # : 3
Document   : Employees receive formal performance reviews twice a year and informal quarterly check-ins.
--------------------------------------------------------------------------------


In [6]:
def rerank_documents(query, documents, top_n=5):
    response = co.rerank(
        model="rerank-v4.0-pro",
        query=query,
        documents=documents,
        top_n=top_n
    )

    return [
        {
            "rank": rank,
            "index": result.index,
            "score": result.relevance_score,
            "document": documents[result.index]
        }
        for rank, result in enumerate(response.results, start=1)
    ]


results = rerank_documents(
    "What are the health benefits?",
    documents,
    top_n=3
)

for result in results:
    print(
        f"{result['rank']}. "
        f"{result['score']:.4f} - "
        f"{result['document']}"
    )


1. 0.7458 - Our health and wellness program includes gym memberships, on-site yoga classes, mental wellness programs, and health insurance.
2. 0.5560 - The company provides flexible working hours and remote-work options for eligible employees.
3. 0.4938 - Employees can claim reimbursement for approved business travel expenses.


The important difference is that the original search may retrieve documents using keyword or vector similarity, while the reranker evaluates the relationship between the complete query and each retrieved document and produces a new **ordering**